In [ ]:
import numpy as np
from skimage import measure
from shapely.geometry import Polygon
from shapely.ops import unary_union
from collections import defaultdict
from PIL import Image
from ecpo_pipeline.cover import *
from ecpo_pipeline.detect import overlay_outline, overlay_solid

In [ ]:
def polygons_from_rgb_array(arr, skip_colors=((0, 0, 0),)):
    """
    Convert an RGB NumPy array into Shapely polygons grouped by RGB value.
    """
    polygons_by_color = defaultdict(list)

    # Get unique RGB values
    colors = np.unique(arr.reshape(-1, 3), axis=0)

    for color in colors:
        color = tuple(int(c) for c in color)
        if color in skip_colors:
            continue

        # Create mask for this RGB value
        mask = np.all(arr == color, axis=-1)

        # Extract contours
        contours = measure.find_contours(mask.astype(np.uint8), level=0.5)

        for contour in contours:
            coords = [(float(c[1]), float(c[0])) for c in contour]

            if len(coords) >= 3:
                poly = Polygon(coords)

                if poly.is_valid and poly.area > 0:
                    polygons_by_color[color].append(poly)

    # # Merge polygons per color
    # for color, polys in polygons_by_color.items():
    #     polygons_by_color[color] = unary_union(polys)

    return polygons_by_color

In [ ]:
img = Image.open(
    "/home/dkempf/ecpo-new-pipeline/download_data/jb_0015_1919-04-15_0002to0003.png"
)
limg = Image.open(
    "/home/dkempf/ecpo-new-pipeline/download_data/layout/jb_0015_1919-04-15_0002to0003_layout.png"
)

In [ ]:
arr = np.array(img)
larr = np.array(limg)

In [ ]:
polys = polygons_from_rgb_array(larr)

In [ ]:
polys = list(reversed(sorted(polys[(0, 0, 255)], key=lambda p: p.area)))

In [ ]:
result = []
for p in polys:
    # There are ridiculously small polygons (down to 1px)
    if p.area < 20:
        continue

    crop, mask, (xoff, yoff) = crop_polygon(arr, p)
    res = layout_detection(crop)

    for r in res:
        poly = p.intersection(translate(r, xoff=xoff, yoff=yoff))
        if poly.area > 20:
            result.append(poly)

In [ ]:
overlay_outline(Image.fromarray(arr), {"text_polys": polys, "image_polys": []})

In [ ]:
overlay_outline(Image.fromarray(arr), {"text_polys": result, "image_polys": []})